In [ ]:
"""
03_test_model_desktop.py
========================
Live webcam testing of the trained BIM model on desktop.
Matches the phone + server pipeline exactly:
  - User visibility check (face + shoulders + elbows required)
  - Motion-aware prediction (static fast / dynamic wait-for-arc)
  - Same normalization, strides, and buffer constants as 04_fastapi_server.py

CONSTANTS — must match 02_train_model.py and 04_fastapi_server.py:
  SEQUENCE_LENGTH = 8
  FEATURE_SIZE    = 204
  STRIDES         = [1, 2]
  RAW_BUFFER_LEN  = 16
"""

import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from collections import deque
import time
import json

# ==============================
# CONFIG  — must match 02_train_model.py
# ==============================
MODEL_PATH  = "bim_model/handface_pose_cnn_lstm.h5"
LABEL_PATH  = "bim_model/labels.json"

SEQUENCE_LENGTH = 8
HAND_FEATURES   = 126
FACE_FEATURES   = 60
POSE_FEATURES   = 18
FEATURE_SIZE    = HAND_FEATURES + FACE_FEATURES + POSE_FEATURES  # 204

STRIDES                 = [1, 2]
RAW_BUFFER_LEN          = max(STRIDES) * SEQUENCE_LENGTH  # 16
PRED_EVERY_N_RAW_FRAMES = 2
CONFIDENCE_THRESHOLD    = 0.65
MIN_FRAME_NONZERO_RATIO = 0.45
SHOW_FPS                = True
MIN_FRAMES_TO_START     = 2   # start predicting after 2 frames (padded to 8)

# Face + pose indices — must match server and data collection
SELECTED_FACE_IDX = [
    13, 14, 78, 308, 82, 312, 33, 133, 362, 263,
    70, 63, 105, 66, 107, 336, 296, 334, 293, 300
]
POSE_IDX = [11, 13, 15, 12, 14, 16]  # shoulders, elbows, wrists

# ==============================
# MOTION-AWARE CONFIG  — identical to 04_fastapi_server.py
# ==============================
STATIC_THRESHOLD      = 0.015  # avg wrist displacement/frame → holding still
DYNAMIC_THRESHOLD     = 0.030  # avg wrist displacement/frame → actively moving
STATIC_CONFIRM_FRAMES = 4      # still frames before static prediction fires
DYNAMIC_SMOOTHING     = 5      # vote window for dynamic signs
STATIC_SMOOTHING      = 2      # vote window for static signs
VELOCITY_HISTORY_LEN  = 6

# ==============================
# VISIBILITY CHECK  — identical to 04_fastapi_server.py
# ==============================
VISIBILITY_THRESHOLD = 0.3   # lenient — partially visible landmarks still count
# Shoulders only — elbows excluded (unreliable visibility scores when arms at rest)
_REQUIRED_POSE_SLOTS = [0, 3]  # left shoulder + right shoulder only
_SLOT_NAMES          = {0: "L.shoulder", 3: "R.shoulder"}

def check_user_visibility(results):
    """Returns (is_visible: bool, reason: str)."""
    if not results.face_landmarks:
        return False, "Face not detected"
    if not results.pose_landmarks:
        return False, "Upper body not detected"
    missing = []
    for slot in _REQUIRED_POSE_SLOTS:
        lm  = results.pose_landmarks.landmark[POSE_IDX[slot]]
        vis = getattr(lm, 'visibility', 1.0)
        if vis < VISIBILITY_THRESHOLD:
            missing.append(_SLOT_NAMES[slot])
    if missing:
        return False, f"Not visible: {', '.join(missing)}"
    return True, "ok"

# ==============================
# NORMALIZATION  — identical to training
# ==============================
def normalize_hand(f):
    f = f.copy()
    xs, ys = f[0::3], f[1::3]
    valid  = xs != 0
    if not valid.any(): return f
    min_x, max_x = xs[valid].min(), xs[valid].max()
    min_y, max_y = ys[valid].min(), ys[valid].max()
    scale = max(max_x - min_x, max_y - min_y) or 1
    for i in range(0, len(f), 3):
        f[i]     = (f[i]     - min_x) / scale
        f[i + 1] = (f[i + 1] - min_y) / scale
    return f

def normalize_hand_pair(h):
    return np.concatenate([normalize_hand(h[:63]), normalize_hand(h[63:])])

def normalize_face(f):
    f = f.copy()
    xs, ys = f[0::3], f[1::3]
    valid  = xs != 0
    if not valid.any(): return f
    cx, cy = xs[0], ys[0]
    scale  = max(xs[valid].max() - xs[valid].min(),
                 ys[valid].max() - ys[valid].min()) or 1
    for i in range(0, len(f), 3):
        f[i]     = (f[i]     - cx) / scale
        f[i + 1] = (f[i + 1] - cy) / scale
    return f

def normalize_pose(f):
    f = f.copy()
    xs, ys = f[0::3], f[1::3]
    valid  = xs != 0
    if not valid.any(): return f
    cx, cy = xs[0], ys[0]
    scale  = max(xs[valid].max() - xs[valid].min(),
                 ys[valid].max() - ys[valid].min()) or 1
    for i in range(0, len(f), 3):
        f[i]     = (f[i]     - cx) / scale
        f[i + 1] = (f[i + 1] - cy) / scale
    return f

# ==============================
# MULTI-STRIDE INFERENCE
# ==============================
def _pad_sequence(frames: list, target_len: int) -> np.ndarray:
    """Left-pad by repeating the first frame until target_len is reached."""
    if len(frames) >= target_len:
        return np.array(frames[-target_len:], dtype=np.float32)
    pad_count = target_len - len(frames)
    return np.array([frames[0]] * pad_count + frames, dtype=np.float32)

def predict_multiscale(rawbuf):
    """Stride=1 runs from MIN_FRAMES_TO_START with left-padding.
    Stride=2 requires full buffer. Returns (idx, conf, stride) or (None,None,None)."""
    best_idx, best_conf, best_stride = None, -1.0, None
    buf = list(rawbuf)

    for stride in STRIDES:
        needed = stride * SEQUENCE_LENGTH
        if stride == 1:
            if len(buf) < MIN_FRAMES_TO_START: continue
            tail = buf[-needed:] if len(buf) >= needed else buf
            seq  = _pad_sequence(tail, SEQUENCE_LENGTH)
        else:
            if len(buf) < needed: continue
            tail = buf[-needed:]
            seq  = np.array(tail[::stride], dtype=np.float32)
            if len(seq) != SEQUENCE_LENGTH: continue

        probs = model.predict(np.expand_dims(seq, 0), verbose=0)[0]
        idx   = int(np.argmax(probs))
        conf  = float(probs[idx])
        if conf > best_conf:
            best_conf, best_idx, best_stride = conf, idx, stride
    return (best_idx, best_conf, best_stride) if best_idx is not None else (None, None, None)

# ==============================
# MOTION STATE MACHINE
# Mirrors SessionState in 04_fastapi_server.py
# ==============================
class MotionState:
    def __init__(self):
        self.velocity_history = deque(maxlen=VELOCITY_HISTORY_LEN)
        self.prev_wrist       = None
        self.prev_spread      = None    # previous fingertip spread value
        self.still_count      = 0
        self.state            = 'static'
        self.arc_started      = False
        self.smooth_buffer    = deque(maxlen=STATIC_SMOOTHING)
        self.buffer_cleared   = False

    def update(self, kp: np.ndarray) -> float:
        """
        Combined velocity = wrist displacement + fingertip spread change.
        Catches both wrist-moving signs (dynamic) and same-wrist
        finger-reconfiguring signs (e.g. A/S/E).
        """
        FINGERTIP_OFFSETS = [12, 24, 36, 48, 60]  # thumb/index/middle/ring/pinky tip x-offsets

        def _wrist_xy(base):
            x, y = kp[base], kp[base + 1]
            return np.array([x, y]) if (x != 0 or y != 0) else None

        def _spread(base):
            tips = []
            for off in FINGERTIP_OFFSETS:
                x, y = kp[base + off], kp[base + off + 1]
                if x != 0 or y != 0:
                    tips.append(np.array([x, y]))
            if len(tips) < 2: return 0.0
            return float(np.mean([
                np.linalg.norm(tips[i] - tips[j])
                for i in range(len(tips))
                for j in range(i + 1, len(tips))
            ]))

        right_wrist = _wrist_xy(63)
        left_wrist  = _wrist_xy(0)

        if right_wrist is not None:
            wrist  = right_wrist
            spread = _spread(63)
        elif left_wrist is not None:
            wrist  = left_wrist
            spread = _spread(0)
        else:
            self.velocity_history.append(0.0)
            self._tick(0.0)
            return 0.0

        wrist_vel  = float(np.mean(np.abs(wrist - self.prev_wrist))) \
                     if self.prev_wrist is not None else 0.0
        spread_vel = abs(spread - self.prev_spread) \
                     if self.prev_spread is not None else 0.0

        self.prev_wrist  = wrist
        self.prev_spread = spread

        velocity = 0.6 * wrist_vel + 0.4 * spread_vel
        self.velocity_history.append(velocity)
        self._tick(velocity)
        return velocity

    def _tick(self, vel: float):
        avg       = float(np.mean(self.velocity_history)) if self.velocity_history else 0.0
        prev_state = self.state

        if avg > DYNAMIC_THRESHOLD:
            self.state       = 'dynamic'
            self.arc_started = True
            self.still_count = 0
            # Sign transition: clear old frames so next sign gets clean buffer
            if prev_state in ('static', 'settling'):
                self.buffer_cleared = True
            if self.smooth_buffer.maxlen != DYNAMIC_SMOOTHING:
                self.smooth_buffer = deque(list(self.smooth_buffer), maxlen=DYNAMIC_SMOOTHING)
        elif avg < STATIC_THRESHOLD:
            self.still_count += 1
            if self.arc_started and self.still_count >= STATIC_CONFIRM_FRAMES:
                self.state       = 'settling'
                self.arc_started = False
            elif not self.arc_started:
                self.state = 'static'
            if self.smooth_buffer.maxlen != STATIC_SMOOTHING:
                self.smooth_buffer = deque(
                    list(self.smooth_buffer)[-STATIC_SMOOTHING:],
                    maxlen=STATIC_SMOOTHING)
        else:
            self.still_count = 0

    @property
    def should_predict(self) -> bool:
        if self.state == 'static':   return self.still_count >= STATIC_CONFIRM_FRAMES
        if self.state == 'settling': return True
        return False

    def mark_settling_done(self):
        self.state       = 'static'
        self.still_count = 0

# ==============================
# LOAD MODEL & LABELS
# ==============================
print("🔄 Loading model...")
model  = tf.keras.models.load_model(MODEL_PATH)
with open(LABEL_PATH, "r") as f:
    labels = json.load(f)
print(f"✅ Model loaded | Classes ({len(labels)}): {labels}")

# ==============================
# MEDIAPIPE
# ==============================
mp_holistic = mp.solutions.holistic
mp_drawing  = mp.solutions.drawing_utils
mp_styles   = mp.solutions.drawing_styles

# Motion state → overlay colour
STATE_COLOR = {
    'static'  : (180, 180, 180),
    'dynamic' : (0,   140, 255),
    'settling': (0,   220,   0),
}

# ==============================
# MAIN LOOP
# ==============================
raw_buffer      = deque(maxlen=RAW_BUFFER_LEN)
motion          = MotionState()
accepted_frames = 0
prev_time       = 0.0
current_sign    = None
current_conf    = 0.0
current_stride  = None

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise SystemExit("❌ Cannot open webcam.")

print("\n🎥 Starting live prediction. Press ESC to quit.\n")
print("  Required to predict : face + both shoulders + both elbows")
print("  Static sign  → predict after 4 still frames (~133ms at 30fps)")
print("  Dynamic sign → predict when motion arc completes\n")

with mp_holistic.Holistic(
    static_image_mode=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    model_complexity=1,
) as holistic:

    while True:
        ret, frame = cap.read()
        if not ret: break

        frame   = cv2.flip(frame, 1)
        h, w    = frame.shape[:2]
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        # FPS
        curr_time = time.time()
        fps       = 1 / (curr_time - prev_time) if prev_time else 0
        prev_time = curr_time

        # ── Draw landmarks ──────────────────────────────────────────────────
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(frame, results.left_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
                mp_styles.get_default_hand_landmarks_style())
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(frame, results.right_hand_landmarks,
                mp_holistic.HAND_CONNECTIONS,
                mp_styles.get_default_hand_landmarks_style())
        if results.face_landmarks:
            for idx in SELECTED_FACE_IDX:
                lm = results.face_landmarks.landmark[idx]
                cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 2, (0,255,0), -1)
        if results.pose_landmarks:
            for slot, pose_idx in enumerate(POSE_IDX):
                lm  = results.pose_landmarks.landmark[pose_idx]
                vis = getattr(lm, 'visibility', 1.0)
                # Required landmark: bright if visible, red if missing
                if slot in _REQUIRED_POSE_SLOTS:
                    c = (255, 100, 0) if vis >= VISIBILITY_THRESHOLD else (0, 0, 255)
                    cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 7, c, -1)
                    cv2.putText(frame, _SLOT_NAMES[slot],
                                (int(lm.x * w) + 8, int(lm.y * h)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.38, c, 1)
                else:
                    cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 4, (180, 80, 0), -1)

        # ── Visibility check ────────────────────────────────────────────────
        user_visible, visibility_reason = check_user_visibility(results)

        # ── Extract keypoints ───────────────────────────────────────────────
        kp = np.zeros(FEATURE_SIZE, dtype=np.float32)
        for h_idx, hand in enumerate([results.left_hand_landmarks,
                                       results.right_hand_landmarks]):
            if hand:
                base = h_idx * 63
                for i, lm in enumerate(hand.landmark):
                    kp[base + i*3]     = lm.x
                    kp[base + i*3 + 1] = lm.y
                    kp[base + i*3 + 2] = lm.z
        if results.face_landmarks:
            for i, idx in enumerate(SELECTED_FACE_IDX):
                lm  = results.face_landmarks.landmark[idx]
                off = HAND_FEATURES + i * 3
                kp[off], kp[off+1], kp[off+2] = lm.x, lm.y, lm.z
        if results.pose_landmarks:
            ps = HAND_FEATURES + FACE_FEATURES
            for i, idx in enumerate(POSE_IDX):
                lm  = results.pose_landmarks.landmark[idx]
                off = ps + i * 3
                kp[off], kp[off+1], kp[off+2] = lm.x, lm.y, lm.z

        # ── Motion update ────────────────────────────────────────────────────
        velocity = motion.update(kp)

        # ── Buffer flush on sign transition ──────────────────────────────────
        if motion.buffer_cleared:
            raw_buffer.clear()
            motion.smooth_buffer.clear()
            motion.buffer_cleared = False
            current_sign = None

        # ── Buffer gate ──────────────────────────────────────────────────────
        if user_visible:
            nonzero = np.count_nonzero(kp) / kp.size
            if nonzero >= MIN_FRAME_NONZERO_RATIO:
                kp_norm = np.concatenate([
                    normalize_hand_pair(kp[:HAND_FEATURES]),
                    normalize_face(kp[HAND_FEATURES:HAND_FEATURES + FACE_FEATURES]),
                    normalize_pose(kp[HAND_FEATURES + FACE_FEATURES:]),
                ])
                raw_buffer.append(kp_norm)
                accepted_frames += 1
        else:
            if len(raw_buffer) > 0:
                raw_buffer.clear()
                motion.smooth_buffer.clear()
                motion.still_count = 0
                current_sign = None

        # ── Motion-aware inference ───────────────────────────────────────────
        min_ready = MIN_FRAMES_TO_START
        ready     = len(raw_buffer) >= min_ready and user_visible

        if ready and motion.should_predict and accepted_frames % PRED_EVERY_N_RAW_FRAMES == 0:
            best_idx, best_conf, best_stride = predict_multiscale(raw_buffer)
            if best_idx is not None and best_conf >= CONFIDENCE_THRESHOLD:
                motion.smooth_buffer.append(best_idx)
                majority       = int(np.bincount(
                    np.array(motion.smooth_buffer, dtype=np.int32)).argmax())
                current_sign   = labels[majority]
                current_conf   = best_conf
                current_stride = best_stride
                if motion.state == 'settling':
                    motion.mark_settling_done()
                    motion.smooth_buffer.clear()
            else:
                motion.smooth_buffer.clear()
                current_sign = None

        # ── UI overlay ───────────────────────────────────────────────────────
        # FPS
        if SHOW_FPS:
            cv2.putText(frame, f"FPS: {int(fps)}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 200, 0), 2)

        # Buffer bar
        bar_filled = int((len(raw_buffer) / RAW_BUFFER_LEN) * 200)
        bar_color  = (0, 220, 120) if ready else (0, 140, 255)
        cv2.rectangle(frame, (10, 45), (210, 58), (60, 60, 60), -1)
        cv2.rectangle(frame, (10, 45), (10 + bar_filled, 58), bar_color, -1)
        cv2.putText(frame, f"Buffer {len(raw_buffer)}/{RAW_BUFFER_LEN}",
                    (10, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

        # Motion state
        m_color = STATE_COLOR.get(motion.state, (180, 180, 180))
        m_label = {'static': 'Still', 'dynamic': 'Moving', 'settling': 'Predict!'
                   }.get(motion.state, motion.state)
        cv2.putText(frame, f"{m_label}  v={velocity:.3f}",
                    (10, 78), cv2.FONT_HERSHEY_SIMPLEX, 0.6, m_color, 2)

        # Prediction
        if current_sign:
            txt = f"{current_sign}  {current_conf*100:.1f}%"
            if current_stride: txt += f"  s={current_stride}"
            cv2.putText(frame, txt,
                        (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)
        elif not user_visible:
            cv2.putText(frame, "Position yourself properly",
                        (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 100, 255), 2)
        else:
            cv2.putText(frame, "Waiting...",
                        (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (100, 100, 100), 1)

        # Visibility status bar at bottom
        vis_color = (0, 160, 0) if user_visible else (0, 80, 200)
        vis_text  = "✓ Properly framed" if user_visible else f"⚠  {visibility_reason}"
        cv2.rectangle(frame, (0, h - 36), (w, h), vis_color, -1)
        cv2.putText(frame, vis_text, (10, h - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        cv2.imshow("BIM Live Prediction", frame)
        if cv2.waitKey(1) == 27:
            break

cap.release()
cv2.destroyAllWindows()
print("✅ Done.")

🔄 Loading model...
✅ Model loaded | Classes (4): ['A', 'B', 'C', 'E']

🎥 Starting live prediction. Press ESC to quit.

  Required to predict : face + both shoulders + both elbows
  Static sign  → predict after 4 still frames (~133ms at 30fps)
  Dynamic sign → predict when motion arc completes

✅ Done.
